# ASD Prediction System - Model Training Pipeline

This notebook demonstrates the complete pipeline for training and evaluating the ASD prediction model.

## Contents
1. Setup and Data Loading
2. Data Preprocessing
3. Feature Engineering
4. Model Training
5. Hyperparameter Tuning
6. Model Evaluation
7. Feature Importance Analysis
8. Save Model and Artifacts

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
import yaml

warnings.filterwarnings('ignore')

# Set up paths
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

# Import custom modules
from data_processing.data_loader import DataLoader, load_config
from data_processing.preprocessor import DataPreprocessor
from feature_engineering.feature_engineer import FeatureEngineer
from models.xgboost_model import ASDXGBoostModel
from evaluation.evaluator import ModelEvaluator

print(f"Project root: {project_root}")
print("All modules imported successfully!")

## 1. Setup and Data Loading

In [ ]:
# Load configuration
config_path = project_root / 'config' / 'config.yaml'
config = load_config(config_path)

print("Configuration loaded:")
print(f"  - Test size: {config['data_processing']['test_size']}")
print(f"  - Random state: {config['data_processing']['random_state']}")
print(f"  - Target metrics: {config['evaluation']['target_performance']}")

In [ ]:
# Load training and test data
data_path = project_root / 'data' / 'raw'
loader = DataLoader(data_path)

train_df = loader.load_raw_data('asd_train_data.csv')
test_df = loader.load_raw_data('asd_test_data.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

# Validate data
is_valid, report = loader.validate_data(train_df)
print(f"\nData validation passed: {is_valid}")
if report['warnings']:
    print(f"Warnings: {report['warnings']}")

## 2. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor(config=config.get('data_processing', {}))

# Separate features and target
target_col = 'asd_diagnosis'
id_col = 'participant_id'

# Keep IDs for later reference
train_ids = train_df[id_col] if id_col in train_df.columns else None
test_ids = test_df[id_col] if id_col in test_df.columns else None

# Drop ID column for processing
if id_col in train_df.columns:
    train_df = train_df.drop(columns=[id_col])
if id_col in test_df.columns:
    test_df = test_df.drop(columns=[id_col])

print(f"Features for preprocessing: {len(train_df.columns) - 1}")

In [ ]:
# Fit preprocessor on training data and transform
X_train, y_train = preprocessor.preprocess_pipeline(
    train_df,
    target_column=target_col,
    missing_strategy='median',
    normalize=False,  # Will normalize after feature engineering
    handle_outliers=True,
    create_interactions=False  # Will do in feature engineering
)

print(f"\nTraining data after preprocessing:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Missing values: {X_train.isnull().sum().sum()}")

In [ ]:
# Transform test data using fitted preprocessor
X_test, y_test = preprocessor.preprocess_pipeline(
    test_df,
    target_column=target_col,
    missing_strategy='median',
    normalize=False,
    handle_outliers=True
)

print(f"Test data after preprocessing:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")

## 3. Feature Engineering

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer(config=config.get('features', {}))

# Apply feature engineering
X_train_eng = feature_engineer.engineer_all_features(X_train)
X_test_eng = feature_engineer.engineer_all_features(X_test)

print(f"Features after engineering:")
print(f"  X_train: {X_train.shape[1]} -> {X_train_eng.shape[1]}")
print(f"  X_test: {X_test.shape[1]} -> {X_test_eng.shape[1]}")

In [ ]:
# Select only numeric columns for modeling
numeric_cols = X_train_eng.select_dtypes(include=[np.number]).columns.tolist()

X_train_final = X_train_eng[numeric_cols].copy()
X_test_final = X_test_eng[numeric_cols].copy()

# Handle any remaining missing values
X_train_final = X_train_final.fillna(X_train_final.median())
X_test_final = X_test_final.fillna(X_train_final.median())

print(f"\nFinal feature set: {len(numeric_cols)} features")
print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

In [ ]:
# Feature selection (optional)
X_selected, importances = feature_engineer.select_features_by_importance(
    X_train_final, y_train,
    method='random_forest',
    n_features=20
)

# Get selected feature names
selected_features = feature_engineer.selected_features
print(f"\nSelected top {len(selected_features)} features:")
for i, feat in enumerate(selected_features[:10], 1):
    print(f"  {i}. {feat}: {importances[feat]:.4f}")

# Use selected features for training
X_train_selected = X_train_final[selected_features]
X_test_selected = X_test_final[selected_features]

## 4. Model Training

In [ ]:
# Initialize XGBoost model with config parameters
model_params = config.get('model', {}).get('xgboost', {})
model = ASDXGBoostModel(params=model_params)

print("Model parameters:")
for key, value in model.params.items():
    print(f"  {key}: {value}")

In [ ]:
# Create validation split from training data
from sklearn.model_selection import train_test_split

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_selected, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

print(f"Training split: {X_train_split.shape[0]} samples")
print(f"Validation split: {X_val.shape[0]} samples")
print(f"Test set: {X_test_selected.shape[0]} samples")

In [ ]:
# Train the model
model.train(X_train_split, y_train_split, X_val=X_val, y_val=y_val)

print("\nModel training complete!")

In [ ]:
# Cross-validation to assess stability
cv_results = model.cross_validate(X_train_selected, y_train, cv=5)

print("\nCross-Validation Results:")
print("="*50)
for metric, value in cv_results.items():
    print(f"  {metric}: {value:.4f}")

## 5. Hyperparameter Tuning (Optional)

In [ ]:
# Define parameter grid (smaller for demonstration)
param_grid = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1, 0.15],
    'n_estimators': [100, 150, 200],
    'subsample': [0.8, 0.9]
}

# Uncomment to run hyperparameter tuning (takes time)
# best_params, best_score = model.hyperparameter_tuning(
#     X_train_selected, y_train,
#     param_grid=param_grid,
#     cv=3
# )
# print(f"\nBest parameters: {best_params}")
# print(f"Best CV score: {best_score:.4f}")

print("Hyperparameter tuning skipped for quick demonstration.")
print("Uncomment the code above to run full tuning.")

## 6. Model Evaluation

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test_selected)
y_prob = model.predict_proba(X_test_selected)

print(f"Predictions made: {len(y_pred)}")
print(f"Prediction distribution: {np.bincount(y_pred)}")

In [ ]:
# Initialize evaluator and generate report
evaluator = ModelEvaluator(config=config.get('evaluation', {}))

# Generate comprehensive evaluation
output_dir = project_root / 'outputs' / 'reports'
report = evaluator.generate_report(
    y_test.values, y_pred, y_prob,
    output_dir=output_dir,
    model_name='ASD_XGBoost_Model_v1'
)

print(f"\nReport saved to: {output_dir}")

In [ ]:
# Print detailed evaluation summary
evaluator.print_summary()

In [ ]:
# Optimize threshold for clinical use
optimal_threshold, optimal_metrics = evaluator.optimize_threshold(
    y_test.values, y_prob,
    metric='f1_score',
    min_sensitivity=0.85
)

print(f"\nOptimal threshold: {optimal_threshold:.3f}")
print(f"Metrics at optimal threshold:")
print(f"  Sensitivity: {optimal_metrics['sensitivity']:.4f}")
print(f"  Specificity: {optimal_metrics['specificity']:.4f}")
print(f"  F1 Score: {optimal_metrics['f1_score']:.4f}")

In [ ]:
# Clinical utility assessment
clinical = evaluator.evaluate_clinical_utility(y_test.values, y_pred, population_size=10000)

print("\nClinical Utility (per 10,000 screened):")
print("="*50)
print(f"  Expected ASD cases: {clinical['expected_cases']}")
print(f"  Cases detected: {clinical['cases_detected']}")
print(f"  Cases missed: {clinical['cases_missed']}")
print(f"  False alarms: {clinical['false_alarms']}")
print(f"  Number needed to screen: {clinical['number_needed_to_screen']}")
print(f"  Referral rate: {clinical['referral_rate']*100:.1f}%")

## 7. Feature Importance Analysis

In [ ]:
# Get feature importances from model
model_importance = model.get_feature_importance()

# Create importance DataFrame
importance_df = pd.DataFrame([
    {'feature': feat, 'importance': imp}
    for feat, imp in model_importance.items()
]).sort_values('importance', ascending=False)

# Display top features
print("Top 15 Most Important Features:")
print("="*50)
for i, row in importance_df.head(15).iterrows():
    print(f"  {row['feature']:<35} {row['importance']:.4f}")

In [ ]:
# Plot feature importance
plt.figure(figsize=(12, 8))
top_features = importance_df.head(15)
sns.barplot(data=top_features, y='feature', x='importance', palette='viridis')
plt.title('Top 15 Feature Importances', fontsize=14)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(output_dir / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Model and Artifacts

In [ ]:
# Save trained model
model_dir = project_root / 'models' / 'trained'
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / 'asd_xgboost_model.joblib'
model.save_model(model_path)
print(f"Model saved to: {model_path}")

In [ ]:
# Save preprocessor
preprocessor_path = model_dir / 'preprocessor.joblib'
preprocessor.save(preprocessor_path)
print(f"Preprocessor saved to: {preprocessor_path}")

In [ ]:
# Save feature engineer
feature_eng_path = model_dir / 'feature_engineer.joblib'
feature_engineer.save(feature_eng_path)
print(f"Feature engineer saved to: {feature_eng_path}")

In [ ]:
# Save selected features list
import json

artifacts = {
    'selected_features': selected_features,
    'optimal_threshold': optimal_threshold,
    'model_metrics': report['metrics'],
    'targets_met': report['targets_met']
}

artifacts_path = model_dir / 'model_artifacts.json'
with open(artifacts_path, 'w') as f:
    json.dump(artifacts, f, indent=2, default=float)

print(f"Artifacts saved to: {artifacts_path}")

In [ ]:
# Summary
print("\n" + "="*70)
print("MODEL TRAINING COMPLETE")
print("="*70)
print(f"\nModel Performance:")
print(f"  - Accuracy: {report['metrics']['accuracy']:.4f}")
print(f"  - Sensitivity: {report['metrics']['sensitivity']:.4f}")
print(f"  - Specificity: {report['metrics']['specificity']:.4f}")
print(f"  - ROC-AUC: {report['metrics']['roc_auc']:.4f}")
print(f"\nTarget Performance:")
for metric, met in report['targets_met'].items():
    status = "✓" if met else "✗"
    print(f"  - {metric}: {status}")
print(f"\nSaved Artifacts:")
print(f"  - Model: {model_path}")
print(f"  - Preprocessor: {preprocessor_path}")
print(f"  - Feature Engineer: {feature_eng_path}")
print(f"  - Artifacts: {artifacts_path}")
print(f"  - Report: {output_dir / 'evaluation_report.json'}")
print("="*70)